# 16 — Docker do serving de laboratório

A referência congelada foi exportada pelo autor e validada por HTTP em um wheel
instalado fora do checkout. Esta etapa empacota o mesmo runtime com dependências
fixadas, modelo externo somente para leitura e identidade verificada na carga.

Não há refit, acesso ao holdout ou consumo de 16–30/09. O produtor de features
causais e a fila de investigação continuam responsabilidades separadas.


## Resultado nativo e alcance

O exportador relatou 67.255 linhas de paridade. Wheel e Docker pontuaram o evento
335910 com score 0.4178630794049542 e tolerância 1e-12. A release conserva ID
`b3deb7e7bb6d4189bc7671453f8643d2` e o modelo congelado.

O autor concluiu Docker em 2026-10-09: inicialização, HTTP 422, reinício, três
falhas de carga, proteções e preservação dos arquivos passaram. Imagem: 166,8 MiB;
snapshot de memória: 178,5 MiB; latência p95: 4,74 ms sob teste sequencial local.

[Resultado, recibo e limites](../../docs/operations/SERVING_CONTRACT.md#resultado-nativo-docker--2026-10-09).
O assistente recebeu saídas de terminal, sem os bytes do modelo nem engine local.


## Execução explícita no host

O [runbook de serving](../../docs/operations/SERVING_CONTRACT.md#construir-e-verificar-docker)
concentra os comandos, proteções, limites e recuperação. O build usa
`docker/serving/Dockerfile`; o verificador está em `scripts/serving/`.

```bash
make build-serving-image
make validate-serving-docker \
  RELEASE_PATH="$PWD/data/serving/reference-v1" \
  MANIFEST_SHA256="1dd762f7e25fde17ea83a00df5ab489e6885b85f0acd10a1a47da8804bed3263"
```

Preserve o recibo aprovado `data/serving/docker-check.json`. Em uma nova execução,
use `DOCKER_REPORT=data/serving/docker-check-02.json` para não sobrescrevê-lo.
Confira as permissões da release conforme o runbook. O check usa a imagem por ID,
confere HTTP/health/readiness, ausência da stack offline, root/modelo read-only,
reinício e falhas de SHA/modelo alterado/modelo ausente. Falhas interrompem o check;
nenhuma saída positiva é fabricada.


In [ ]:
import json
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
report_path = root / "data/serving/docker-check.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    result = {
        key: report[key]
        for key in (
            "status",
            "image",
            "evidence_kind",
            "http_score",
            "restart_recovery",
            "failure_scenarios",
            "resources",
            "latency",
        )
    }
else:
    result = {"status": "awaiting_native_docker_execution", "report_path": str(report_path)}
result

## Interpretar as medições e avançar

Os percentis usam 100 requisições sequenciais ao mesmo caso depois de dez de
aquecimento. Incluem transporte HTTP local; não demonstram carga concorrente ou
SLA. A memória/CPU é um snapshot, sem pico medido. A imagem contém dependências
`main`, sem Poetry ou dados; o modelo permanece externo e identificado por hash.

As correções do engine e dos logs permitiram identificar a falta de leitura do
manifesto pelo UID 10001. O ajuste de permissões preservou os hashes; o check
nativo seguinte passou. `httpx2` no grupo `dev` resolve a depreciação do TestClient,
com um check estrito antes da validação final.

Fechar o PR Docker e conferir sua CI antecedem a branch Prefect. O primeiro fluxo
verificará release e serviço e publicará um recibo idempotente, com retries apenas
em falhas transitórias. Replay causal e feedback atrasado exigem contrato próprio;
16–30/09 continua reservado. O [mural](../../docs/project/ROADMAP.md#próxima-entrega-concreta)
mantém a sequência e os critérios de entrega.
